# 第14讲：企业知识与共享经验

使用 Hermes 原生 OpenViking Provider。先看无费用的消息转换，再观察真实 Agent 检索、同步回读、显式提交与第二身份召回。原始消息保存、提交被接受、经验已经可用，是三个不同的结果。

先按 README 准备 Hermes 与独立 OpenViking 服务。Notebook 不安装或导入 OpenViking；付费部分默认关闭，源文件不含运行输出。

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
if not (here / 'native_notebook.py').exists():
    here = here / 'examples/14-external-memory-provider'
assert (here / 'native_notebook.py').is_file(), '请从仓库根目录或本讲目录打开 Notebook'
sys.path.insert(0, str(here))
from native_notebook import run_hermes

ALLOW_LIVE = False
ALLOW_COMMIT = False
# 凭据只通过启动 Notebook 的进程环境传入，不写在代码格里。


## 1. 按调用ID转换新信息

这是构造输入，不是实际完成过的调查。检查 `check-1` 的参数与结果是否配对，`recall-1` 及检索返回的旧内容是否被排除。

In [ ]:
conversion_run = run_hermes('''
import json
from native_openviking import HERE, load_native, demo_messages
Manager, Provider = load_native(HERE / 'runtime/native/offline-home')
batch = Provider._messages_to_openviking_batch(
    demo_messages(), assistant_peer_id='investigator-A')
print(json.dumps(batch, ensure_ascii=False, indent=2))
''')
print((conversion_run / 'process.log').read_text())


## 2. 查看服务配置

服务必须使用独立 `runtime/native/venv`，在另一个终端启动并按 `Ctrl-C` 停止。这里仅查看无密钥配置：GLM-5.2、Embedding-3、8000输出上限，以及包含 `experiences` 的会话策略。生成配置前可设置 `GLM_BASE_URL` 和 `GLM_EMBEDDING_BASE_URL`。

In [ ]:
import json
from openviking_config import build_config
from native_openviking import MEMORY_POLICY
print(json.dumps(build_config(), ensure_ascii=False, indent=2))
print(json.dumps(MEMORY_POLICY, ensure_ascii=False, indent=2))


## 3. 让A检索资料，再确认消息保存

确认服务已启动、接受模型费用后，将 `ALLOW_LIVE` 改成 `True`。`NativeSession(agent_runtime=True)` 会让真实 AIAgent 按 `memory.provider: openviking` 加载适配器，下面直接调用 `run_conversation()`。同步发生在原生收尾链路；`verify_write()` 只排空并回读，不代为同步或补写答案。

只有需要本轮进入经验抽取时，才同时打开 `ALLOW_COMMIT`。没有真正的新经验也可能不产生记忆，不能要求模型每次都制造一条。

In [ ]:
if ALLOW_LIVE:
    a_run = run_hermes(f'COMMIT = {ALLOW_COMMIT!r}\n' + '''
from native_openviking import HERE, NativeSession, require_completed
with NativeSession(allow_live=True, agent_runtime=True, agent='investigator-A') as session:
    added = session.tool('viking_add_resource',
        url=str(HERE / 'resources/report-runbook.md'), wait=True, timeout=120)
    session.save('resource-receipt.json', added)
    query = '根据企业资料给出报表延迟排查顺序，引用资料地址，说明需要现场核实的条件。'
    result = session.ai.run_conversation(user_message=query, conversation_history=[])
    session.save('conversation.json', result)
    require_completed(result)
    receipt = session.verify_write(result['messages'], user_content=query,
        assistant_content=result['final_response'])
    session.save('write-receipt.json', receipt)
    if COMMIT:
        session.end_session()
    print(session.home)
''', timeout=420)
    print('运行日志：', a_run / 'process.log')
else:
    print('跳过真实服务与模型调用')


## 4. 沿实际URI逐层读取

从A的检索回执复制一个实际返回的 URI，在下面比较摘要、概览和原文。这是直接接口实验，不是模型自主选择读取层级的证明。

In [ ]:
RESOURCE_URI = ''
if ALLOW_LIVE and RESOURCE_URI:
    read_run = run_hermes(f'URI = {RESOURCE_URI!r}\n' + '''
import json
from native_openviking import NativeSession
with NativeSession(allow_live=True) as session:
    for level in ('abstract', 'overview', 'full'):
        result = session.tool('viking_read', uri=URI, level=level)
        print(json.dumps({'level': level, 'result': result}, ensure_ascii=False))
''')
    print('分层读取日志：', read_run / 'process.log')
else:
    print('尚未启用真实调用或尚未填写实际URI')


## 5. B使用独立Session

在服务完成A的抽取与索引后运行。B与A共享 account/user，但 peer不同，也不传入A的对话历史。检查它实际召回的内容和引用：找到同一份手册不等于使用了A的新经验。模型若没有调用搜索工具，也要区分是否已通过 prefetch取得资料，不能编造主动检索回执。

In [ ]:
if ALLOW_LIVE:
    b_run = run_hermes('''
from native_openviking import NativeSession
with NativeSession(allow_live=True, agent_runtime=True, agent='investigator-B') as session:
    report = session.converse('报表再次延迟。结合可检索资料与既有经验给出排查顺序，说明必须重新核实的条件。')
    print(session.home)
    print('同步回读：', report['write_receipt']['write_verified'])
''', timeout=420)
    print('B运行日志：', b_run / 'process.log')
else:
    print('跳过B的真实调用')


## 观察与停止

客户端每格使用独立进程和 `with` 清理；中断时执行器先发送中断信号，留出 Provider shutdown 时间，再在超时后强制结束。正常清理不隐式提交会话。服务仍在自己的终端运行，按 `Ctrl-C` 停止。

把前台回答、工具回执、同步回读、commit结果和经验是否可检索分开记录。读失败时前台继续并不代表写入成功。SQLite的真实文件故障对照请单独运行 `test_offline.py`，不要把它当成OpenViking验证。

付费格尚未作为本轮真实结果验证。运行文件在忽略的 `runtime/` 中；公开Notebook前清除输出，不能公开密钥、企业资料或个人会话。